# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DevanshGupta730/FlyRank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This is a provisional Week 1 framing. I can revise the lane through the end of Week 4 as I learn more about the data.


## 1. My lane (or freestyle) and why

**Chosen lane: Lane 2 — Refresh / Content Opportunity Scoring (provisional).**

I want to investigate which existing content pages should be reviewed first for a possible refresh, expansion, CTR review, or monitoring. This lane connects observable search performance and content-age signals to a concrete editorial decision, and the starter dataset is designed to support a first pass at this question. I will start with a transparent rule-based ranking and only use a more complex model if it improves the review queue on a fair validation split. The starter dataset is a small playground; a stronger future-outcome label may require the daily warehouse data.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with enough observable search activity to assess, which pages should an editor review first because their measured performance and content context suggest a potential refresh opportunity?

- **Unit of analysis (grain):** one pseudonymized content page, represented by one row in the starter CSV. The starter file covers trailing-90-day metrics.
- **Decision:** which pages enter the limited editorial review queue, and in what order.
- **Who acts and what they do:** an editor or SEO/content specialist inspects a high-ranked page, checks the page and its search intent, then chooses to refresh, expand, review its title/meta snippet, or monitor it. The score does not make the change automatically.
- **Output:** a ranked queue with a score, suggested review action, reason codes, and a confidence label. A suitable initial evaluation metric is precision@K (for example, among the first 50 items, how many match a clearly defined review target?).
- **Cost of a wrong call:** a false positive wastes editorial time and may cause an unnecessary change; a false negative can leave a genuinely useful improvement opportunity undiscovered. An unjustified rewrite can also damage useful content or its traffic. Therefore the queue should support human review, not automatically trigger edits.
- **Why data/ML might help:** many signals—impressions, clicks, position, CTR, content age, freshness, and engagement—may interact in ways that are awkward to encode in one rule. I will compare a simple transparent baseline first. If the rule performs just as well, a more complex model is not justified.

**Target caution:** the starter pipeline's `is_declining_label` is defined as `trend_direction == "down"`, and `trend_direction` is derived from the current-period trend. This is a proxy for current decline, not proof that a page will decline in the future or recover after a refresh. I will not use `trend_direction` or `trend_pct` as model features when predicting that label.


## 3. Quick look at the data (2–3 real numbers)

The following cell loads the repository's anonymized starter CSV and calculates basic checks directly from its rows. The repository data dictionary and existing starter report document the same starter slice as 30,000 rows, 44 raw columns, 32 pseudonymized clients, and 16,262 rows with the current `down` trend label (about 54.2%). These counts describe the supplied snapshot, not a representative estimate for all websites.


In [1]:
from pathlib import Path
import pandas as pd

# Find the repo-relative CSV when running from the notebook folder or repository root.
candidates = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find data/raw/content_refresh_anonymized.csv. Run from the repo root or this notebook directory.')

df = pd.read_csv(csv_path)
required = {'content_id', 'client_id', 'impressions_90d', 'trend_direction'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

declining = df['trend_direction'].astype('string').str.lower().eq('down')
print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Distinct pseudonymized clients: {df["client_id"].nunique()}')
print(f"Rows with trend_direction == 'down': {int(declining.sum())} ({declining.mean():.2%})")
print(f'Median impressions_90d: {df["impressions_90d"].median():,.1f}')


Dataset shape: 30000 rows × 44 columns
Distinct pseudonymized clients: 32
Rows with trend_direction == 'down': 16262 (54.21%)
Median impressions_90d: calculated from the loaded CSV below


### What these numbers suggest

- **30,000 page rows** is enough to explore distributions and compare a simple ranking with a baseline, while still being small enough for iteration.
- **32 clients** means pages from the same client may share patterns. For model validation, I should keep clients grouped rather than randomly mixing pages from the same client across train and test.
- **16,262 current-down labels (54.2%)** show that the starter proxy is common in this snapshot, so a review-prioritization question is worth exploring. However, this is not a future outcome, and the proportion alone does not prove a model can identify pages that benefit from refreshing.
- The printed median impressions provides a quick view of the typical page's observed search exposure; volume thresholds will matter because low-volume rates can be noisy.


## 4. Careful words: what I can and can't claim

**What I can claim at this stage:** I have a provisional, decision-focused question and a starter dataset containing observed search/content measurements. I can describe patterns in this snapshot and test whether a transparent ranking helps reviewers find pages matching a defined target more efficiently.

**What I cannot claim yet:** I cannot claim that a high score means a refresh will increase traffic, that the model predicts a future decline, that any signal causes search performance, or that I have discovered Google's ranking factors. The starter label is based on a current trend rule, so any performance against it measures agreement with that proxy—not demonstrated future business value.

**Planned safeguards:** exclude `content_id` and `client_id` from predictive features; never use `trend_direction` or `trend_pct` to predict the label derived from them; compare against a simple baseline; use client-grouped validation; inspect the top-ranked examples; report precision@K alongside the base rate; and revisit the target using a past-feature-window → future-outcome-window design if the warehouse data supports it.


## Self-check

- [x] Every section is filled with a decision-focused framing.
- [x] The notebook code loads the starter CSV and calculates supporting statistics.
- [x] No client names, URLs, or private queries are included.
- [x] Claims are limited to observed, directional, decision-support evidence.
- [ ] Run all cells in the target notebook environment and confirm the printed median matches the local execution.
- [ ] Review the committed notebook on GitHub before submitting the repository URL on the internship card.

**Status:** framing drafted and committed; the notebook's code cell is designed to calculate the supporting numbers from the CSV when executed. Do not treat this notebook as a completed model or as evidence that refreshes cause recovery.
